# PDF Metadata, Docling, & Chunks Viewer

This notebook allows you to:
1. Verify that all PDFs in your dataset have a native text layer.
2. Visualize how **Docling** parses the PDFs and splits them into semantic chunks.

In [ ]:
!pip install -q pypdf docling tqdm

In [ ]:
import os
import glob

# Directory containing your PDFs
pdf_dir = "/Users/tri/Projects/arxiv-scholar/nlp_ml_gcs_pdfs"
pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))
pdf_files.sort()

print(f"Found {len(pdf_files)} PDFs in {pdf_dir}")

### 1. Verify All PDFs Have Embedded Text
The code below iterates through all the PDFs and checks if at least 50 characters of text can be extracted from the first page without using OCR.

In [ ]:
import pypdf
from tqdm import tqdm

def verify_all_pdfs_have_text(pdf_list):
    no_text_pdfs = []
    error_pdfs = []
    
    print(f"Verifying {len(pdf_list)} PDFs...\n")
    
    for pdf_path in tqdm(pdf_list, desc="Checking PDFs"):
        try:
            reader = pypdf.PdfReader(pdf_path)
            if len(reader.pages) > 0:
                text = reader.pages[0].extract_text() or ""
                if len(text.strip()) < 50:
                    no_text_pdfs.append(pdf_path)
        except Exception as e:
            error_pdfs.append((pdf_path, str(e)))
            
    print("\n\033[1m--- Verification Results ---\033[0m")
    if not no_text_pdfs and not error_pdfs:
        print("✅ SUCCESS: All PDFs have an embedded text layer!")
    else:
        if no_text_pdfs:
            print(f"❌ Found {len(no_text_pdfs)} PDFs with no/little text (these might be scanned and require OCR).")
            for p in no_text_pdfs[:10]:
                print(f"  - {os.path.basename(p)}")
        if error_pdfs:
            print(f"⚠️ Found {len(error_pdfs)} PDFs that could not be read.")
            
if pdf_files:
    verify_all_pdfs_have_text(pdf_files)

### 2. Docling Chunk Visualizer (OCR Disabled)
This cell picks a random PDF, parses it using Docling, and then uses Docling's `HierarchicalChunker` to break the document down into semantic chunks (grouping by headings, paragraphs, lists, etc). It then prints out the first 10 chunks.

In [15]:
import random
from IPython.display import display, Markdown

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.chunking import HierarchicalChunker

# Set up Docling with OCR Disabled, matching your workflow
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

if pdf_files:
    # Pick a random PDF to parse
    pdf_path = "/Users/tri/Projects/arxiv-scholar/nlp_ml_gcs_pdfs/2605.23572v1.pdf"
    print(f"Parsing {os.path.basename(pdf_path)} with Docling (OCR disabled). This may take a moment...\n")
    
    try:
        # 1. Run Docling conversion
        result = converter.convert(pdf_path)
        
        # 2. Run Docling Chunking
        chunker = HierarchicalChunker()
        chunks = list(chunker.chunk(result.document))
        
        print(f"\033[1m✅ Document parsed and split into {len(chunks)} chunks!\033[0m\n")
        print(f"\033[1m--- Visualizing First 15 Chunks ---\033[0m\n")
        
        # 3. Print out the chunks
        for i, chunk in enumerate(chunks[:15]):
            # We use ANSI colors to make the chunks easy to visually distinguish in the notebook
            print(f"\033[94m\033[1m[ Chunk {i+1} ]\033[0m")
            print(chunk.text)
            print("-" * 80 + "\n")
            
    except Exception as e:
        print(f"Error processing with Docling: {e}")
else:
    print("No PDFs found to display.")

Parsing 2605.23572v1.pdf with Docling (OCR disabled). This may take a moment...



Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1120.30it/s]


✅ Document parsed and split into 152 chunks!

--- Visualizing First 15 Chunks ---

[ Chunk 1 ]
Vipul Gupta, Shikhar Mohan, Lakshya Kumar,
--------------------------------------------------------------------------------

[ Chunk 2 ]
Pranjal Chitale, Nikit Begwani, Amit Singh, Manik Varma Microsoft AI
--------------------------------------------------------------------------------

[ Chunk 3 ]
India
--------------------------------------------------------------------------------

[ Chunk 4 ]
In the competitive landscape of sponsored search, balancing retrieval quality with production latency is a critical challenge. While large retrieval models based on Small Language Models (SLMs) such as Qwen3-Embedding-4B/8B set strong upper bounds on public benchmarks,theirdeploymentinhigh-throughput,latency-sensitive environments remains impractical. In this paper, we present Harness-LM (HLM), a three-phase training framework for transferring the capabilities of large-scale retrievers into compact, 